# 05 · Agent / Controller: the Intelligent Retail Decision Assistant

A **hybrid, inspectable controller** (`src/agent.py`). Its routing is transparent: every decision is written to a trace.

| Tool | What it does | Grounding |
|---|---|---|
| `product_search` | RAG recommendation (notebook 03) | ids must be retrieved; budget enforced |
| `policy_qa` | policy answers (notebook 04) | doc_id citation + verbatim quote, or refusal |
| `segment_insights` | analytics from the RFM segments (notebook 02) | numbers read from `segment_summary.json`, no LLM |
| `review_summary` | what customers say about a product type | counts computed; summary only from review texts |
| `summarise_products` | explain the top 3 of the previous search | session memory, only those 3 ids |
| `calculator` | discounts and arithmetic | safe AST evaluation |

**Routing steps:**
1. **Guardrails** stop empty or out-of-scope input.
2. **Rule-based intent scores** come from regex cues, with specificity rules: for example, "watch" inside a warranty question is not a shopping request.
3. A **multi-intent** query runs up to `MAX_TOOL_CALLS = 3` tools, and each tool receives only its own clause.
4. If no rule fires, an **LLM router** chooses one tool from the fixed list, as JSON, and its choice is validated against the list.

In [1]:
# --- setup: make the project package importable and show the run mode ---
import sys, os, json, warnings
from pathlib import Path
# library deprecation notices print install paths; hide them here and in parallel worker processes
os.environ["PYTHONWARNINGS"] = "ignore"; warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src                                   # sets macOS OpenMP compatibility flags
from src import config
from src.llm import mode_label
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", palette="Set2"); plt.rcParams["figure.dpi"] = 110
OUT = config.OUTPUTS
print("Mode:", mode_label(), "| outputs ->", OUT.relative_to(ROOT))

Mode: LIVE (gpt-4o-mini + text-embedding-3-small) | outputs -> outputs


In [2]:
from src.agent import RetailAssistant, route, score_intents, CUES, TOOLS, SUPPRESS
pd.DataFrame({"tool": list(CUES), "cue patterns": [", ".join(v) for v in CUES.values()]})

,tool,cue patterns
0,policy_qa,"\breturn(s|ed|ing)?\b(?!.*segment), refund, exchange, warranty, guarantee, shipping, deliver(y|ed), how lo..."
1,product_search,"recommend, suggest, \bfind\b, \bshow\b, looking for, \bneed\b, \bwant\b, \bbuy\b, (under|below|budget).*\d..."
2,segment_insights,"segment, cluster, persona, which customers, customer group, price[- ]sensitiv, most valuable customers"
3,review_summary,"review, what do (customers|people|buyers) say, feedback, complain"
4,summarise_products,"summari[sz]e .*(retrieved|above|these|those|top|previous|results), compare (the|these|those) (products|ite..."
5,calculator,"\d+\s*%\s*(off|discount), \d+(\.\d+)?\s*[-+*/x]\s*\d+, \bcalculate\b|\bcompute\b"


## 1. Routing decisions only (no tools executed)

In [3]:
probe = ["Recommend casual sneakers under ₹4000 for daily wear.", "Can I return a discounted product after 7 days?",
         "Is physical damage to my watch covered under the warranty?", "Which customer segment seems most price-sensitive?",
         "Show me a blue backpack for travel under 3000 and tell me if I can exchange it later.",
         "What do customers say about Nike running shoes?", "Summarise the top three retrieved products.",
         "What is 20% off 2499?", "Tell me about the weather today", "Do you offer gift wrapping?"]
rt = []
for q in probe:
    tools, trace = route(q)
    rt.append({"query": q, "selected": tools, "why": " | ".join(trace)})
routes = pd.DataFrame(rt); routes.to_csv(OUT / "05_routing_table.csv", index=False); routes

,query,selected,why
0,Recommend casual sneakers under ₹4000 for daily wear.,[product_search],intent scores: product_search=3 | selected tools: ['product_search']
1,Can I return a discounted product after 7 days?,[policy_qa],intent scores: policy_qa=1 | selected tools: ['policy_qa']
2,Is physical damage to my watch covered under the warranty?,[policy_qa],"intent scores: policy_qa=1, product_search=1 | product noun without shopping intent inside a policy questi..."
3,Which customer segment seems most price-sensitive?,[segment_insights],intent scores: segment_insights=2 | selected tools: ['segment_insights']
4,Show me a blue backpack for travel under 3000 and tell me if I can exchange it later.,"[product_search, policy_qa]","intent scores: policy_qa=1, product_search=3 | selected tools: ['product_search', 'policy_qa']"
5,What do customers say about Nike running shoes?,[review_summary],"intent scores: product_search=1, review_summary=1 | 'review_summary' is more specific -> suppressed ['prod..."
6,Summarise the top three retrieved products.,[summarise_products],intent scores: summarise_products=1 | selected tools: ['summarise_products']
7,What is 20% off 2499?,[calculator],intent scores: calculator=1 | selected tools: ['calculator']
8,Tell me about the weather today,[],"guardrail: out-of-scope topic -> polite refusal, no tool"
9,Do you offer gift wrapping?,[policy_qa],intent scores: none matched | no rule fired -> LLM router chose policy_qa: The request is about store poli...


*"Do you offer gift wrapping?"* matches no rule:
- **Live mode:** the LLM router decides, typically choosing `policy_qa`, which then **refuses** because no policy covers gift wrapping.
- **Offline mode:** the assistant asks the user to rephrase.

## 2. Full assistant demo with traces

This covers the 5 example queries from the brief plus extra cases.

In [4]:
bot = RetailAssistant()
demo = ["Recommend casual sneakers under ₹4000 for daily wear.",
        "Can I return a discounted product after 7 days?",
        "Suggest products for a premium customer who prefers black formal shoes.",
        "Which customer segment seems most price-sensitive?",
        "Summarise the top three retrieved products and explain why they fit the query.",
        "Show me a blue backpack for travel under 3000 and tell me if I can exchange it later.",
        "What do customers say about Nike running shoes?",
        "Is physical damage to my watch covered under the warranty?",
        "What is 20% off 2499?",
        "What's the weather in Mumbai today?"]
outs = [bot.ask(q) for q in demo]

USER: Recommend casual sneakers under ₹4000 for daily wear.
TRACE:
  intent scores: product_search=3
  selected tools: ['product_search']
  step 1: product_search -> status=ok (0.425s)
      constraints parsed: {"max_price": 4000.0, "sub_category": ["Sneakers"]}
      retrieved 5 items (min similarity 0.3): ITM0845(0.54), ITM0905(0.54), ITM0851(0.53), ITM0565(0.53), ITM0282(0.53)
      validation: all recommended items grounded and within budget
ASSISTANT (0.425s):
Recommended products:
  - [ITM0845] Skechers Black Sneakers Synthetic Shoes - Rs 2,745: Designed for daily wear and offers comfort and style.
  - [ITM0905] Adidas Black Sneakers Synthetic Shoes - Rs 2,414: Suitable for daily wear with a good average rating of 4.3/5.
  - [ITM0851] Nike Grey Sneakers Mesh Shoes - Rs 1,919: Offers comfort and style for daily wear with a solid rating of 4.1/5.
  Why: All recommended sneakers are under ₹4000 and designed for daily wear, making them suitable for your request. They come from reputa

In [5]:
with open(OUT / "05_demo_transcript.jsonl", "w") as f:
    for o in outs: f.write(json.dumps(o, default=str, ensure_ascii=False) + "\n")
pd.DataFrame([{"query": o["query"], "tools": o["tools"], "latency_s": o["latency_s"],
               "first line of answer": o["final_answer"].splitlines()[0][:90]} for o in outs])

,query,tools,latency_s,first line of answer
0,Recommend casual sneakers under ₹4000 for daily wear.,[product_search],0.425,Recommended products:
1,Can I return a discounted product after 7 days?,[policy_qa],0.003,Policy: It depends on the discount percentage. Returns are allowed within 7 days for unuse
2,Suggest products for a premium customer who prefers black formal shoes.,[product_search],0.001,Recommended products:
3,Which customer segment seems most price-sensitive?,[segment_insights],0.001,The most price-sensitive (largest conversion lift from deep discounts) segment is **Freque
4,Summarise the top three retrieved products and explain why they fit the query.,[summarise_products],0.000,Summary of the top results:
5,Show me a blue backpack for travel under 3000 and tell me if I can exchange it later.,"[product_search, policy_qa]",0.002,Recommended products:
6,What do customers say about Nike running shoes?,[review_summary],0.002,"Based on 22 matching items: sentiment {'negative': 3, 'positive': 15, 'neutral': 4}, avera"
7,Is physical damage to my watch covered under the warranty?,[policy_qa],0.000,"Policy: No, physical damage to your watch is not covered under the warranty. [sources: POL"
8,What is 20% off 2499?,[calculator],0.000,"20% off Rs 2,499 = Rs 1,999.20 (you save Rs 499.80)."
9,What's the weather in Mumbai today?,[],0.000,"I can help with product recommendations, store policies (returns, refunds, exchange, shipp"


**What the traces show:**
- **Single-intent queries** each hit exactly one tool.
- **Multi-intent:** the backpack + exchange query runs `product_search` on *"blue backpack … under 3000"* and `policy_qa` on *"can I exchange it later"*, then combines the two answers.
- **Session memory:** "summarise the top three" reuses the previous search. It does not search again.
- **Specificity:** "watch … warranty" goes only to `policy_qa`.
- **Guardrail:** "weather" never reaches a tool.
- **Bounded:** at most 3 tool calls per query, each with its own status and latency in the trace.